# EEG-Flow — 1b. Create behavioral metadata (active cohort only)

Computes trial-level behavioral metadata from the TRIGGER channel of the
step1 raw file. Uses MNE's `make_metadata` with a 0–0.999s response window
per stimulus onset. Each trial is classified as one of: Hits, Misses,
CorrectRejections_standard, CorrectRejections_novel, FalseAlarms_standard,
FalseAlarms_novel, FalseAlarms_tooquick (RT < 200ms).

Outputs per run:
- `_step1b_a-metadata.csv` — one row per stimulus event with response
  type and correctness flag; used by step 8 for trial classification
- `_step1b_b-behav.txt` — summary statistics (d', hit rate, FA rate)
- `_step1b_RT.svg` — reaction time histogram (plots/ folder)

Last edit: 14.10.2026 @anguyen

In [ ]:
import multiprocessing as mp
import time
from itertools import product

from mne import set_log_level as set_log_level_mne

from eeg_flow import set_log_level
from eeg_flow.utils import parallel
from eeg_flow.tasks import create_behavioral_metadata

The parameters of the file to process are defined below. Locks are created to prevent someone else from running the same task and from writing the same derivatives.

In [ ]:
set_log_level_mne("WARNING")
set_log_level("INFO")
# set_log_level("DEBUG")

In [ ]:
# PARTICIPANTS_WITH_GROUPS: list[str] = ["P01-G1"]  # list of str "Pxx-Gy", e.g. ["P02-G2"]

# Active (49 participants)
PARTICIPANTS_WITH_GROUPS: list[str] = [
    "P01-G1", "P02-G2", "P03-G1", "P04-G3", "P05-G2", "P06-G4", "P07-G5", "P08-G6",
    "P10-G8", "P11-G1", "P12-G2", "P13-G3", "P14-G3", "P15-G4", "P16-G4", "P17-G5",
    "P18-G6", "P21-G7", "P22-G7", "P23-G7", "P24-G8", "P25-G1", "P26-G2", "P27-G8",
    "P28-G1", "P29-G3", "P30-G4", "P31-G5", "P32-G6", "P33-G7", "P53-G8", "P55-G1",
    "P56-G8", "P58-G5", "P59-G8", "P60-G6", "P61-G4", "P62-G3", "P63-G5", "P64-G3",
    "P65-G1", "P66-G5", "P67-G3", "P69-G8", "P70-G8", "P72-G1", "P73-G3", "P74-G1",
    "P75-G1"]

TASKS: list[str]                    = ["oddball", "UT"]  # ["oddball"], ["UT"] or ["oddball", "UT"]
RUNS: list[int]                     = [1, 2]  # [1], [2] or [1, 2]

inputs = [
    (t[0].split("-") + list(t[1:]))
    for t in product(PARTICIPANTS_WITH_GROUPS, TASKS, RUNS)
]
print(inputs)

In [ ]:
%%time
current_time = time.strftime("%H:%M:%S")
print("Start time", current_time)

assert len(inputs) != 0  # sanity-check
# each job will need 1 threads and mp.cpu_count() usually returns
# the number of threads (usually 2 per core), thus let's spawn 2
# process per core maximum.
n_jobs = min(len(inputs), (mp.cpu_count() - 2))
parallel(create_behavioral_metadata, n_jobs, inputs)